In [3]:
# ============================
# 1. Import thư viện cần thiết
# ============================
import pandas as pd
import numpy as np
from google.colab import files
import io

# Đọc dữ liệu
df = pd.read_csv("/content/Data.csv")

# Xem qua dữ liệu
print(df.shape)
print(df.head())


(3380, 17)
   restaurant_id               restaurant_name                  dish_name  \
0           4740  GUTA CAFE - 107H Trương Định  Bánh mì que pate chà bông   
1           4740  GUTA CAFE - 107H Trương Định           Matcha Chanh Dây   
2           4740  GUTA CAFE - 107H Trương Định         Matcha Phúc Bồn Tử   
3           4740  GUTA CAFE - 107H Trương Định          Ô Long sữa Matcha   
4           4740  GUTA CAFE - 107H Trương Định               Matcha Latte   

                                           dish_desc  price  num_purchases  \
0  Bánh Mì Que Pate Chà Bông sự kết hợp của pate ...   7000              0   
1                                                NaN  35000              0   
2                                                NaN  35000              1   
3                                                NaN  35000              0   
4                                                NaN  35000              2   

   num_likes  num_dislikes restaurant_district        res

In [26]:
#Dùng feature mới
# ============================
# 1. Import thư viện
# ============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import xgboost as xgb

# ============================
# 2. Tạo feature mới (bỏ district_avg_price, price_vs_district)
# ============================
df["price_rating"] = df["price"] * df["restaurant_rating"]
df["likes_ratio"] = df["num_likes"] / (df["num_likes"] + df["num_dislikes"] + 1)
df["rating_weighted"] = df["restaurant_rating"] * df["num_ratings"]
df["log_num_ratings"] = np.log1p(df["num_ratings"])
df["log_num_likes"] = np.log1p(df["num_likes"])

# ============================
# 3. X, y
# ============================
features = [
    "price", "num_likes", "num_dislikes",
    "restaurant_rating", "num_ratings", "avg_price",
    "restaurant_district", "restaurant_type",
    # feature mới
    "price_rating", "likes_ratio", "rating_weighted",
    "log_num_ratings", "log_num_likes"
]

X = df[features]
y = df["num_purchases"]

# log-transform target
y_log = np.log1p(y)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# ============================
# 4. Tiền xử lý
# ============================
numeric_features = [
    "price", "num_likes", "num_dislikes",
    "restaurant_rating", "num_ratings", "avg_price",
    "price_rating", "likes_ratio", "rating_weighted",
    "log_num_ratings", "log_num_likes"
]

categorical_features = ["restaurant_district", "restaurant_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# ============================
# ============================
# 5. Định nghĩa mô hình
# ============================
from sklearn.model_selection import GridSearchCV

# GridSearch cho Random Forest
rf_reg = RandomForestRegressor(random_state=42)
rf_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5]
}

# Gradient Boosting & XGBoost giữ nguyên
gb_reg = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42
)
xgb_reg = xgb.XGBRegressor(
    n_estimators=300, learning_rate=0.1, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)

# ============================
# 6. Huấn luyện & đánh giá
# ============================

# --- GridSearch cho Random Forest ---
print("\n=== GridSearch for Random Forest ===")
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_reg)
])

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print("Best Params:", rf_grid.best_params_)

# Đánh giá RF
y_pred_log = best_rf.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

print("\n=== Random Forest (best) ===")
print("MSE :", mean_squared_error(y_true, y_pred))
print("MAE :", mean_absolute_error(y_true, y_pred))
print("R²  :", r2_score(y_true, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_true, y_pred)*100, "%")

# 6. Gradient Boosting (set thủ công)
# ============================
gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])
gb_model.fit(X_train, y_train)

# ============================
# 7. XGBoost (đơn giản, n_estimators=200)
# ============================
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])
xgb_model.fit(X_train, y_train)

# ============================
# 8. Đánh giá trên test set
# ============================
models = {
    "Random Forest (GridSearch)": best_rf,
    "Gradient Boosting": gb_model,
    "XGBoost": xgb_model
}

for name, model in models.items():
    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)

    print(f"\n=== {name} ===")
    print(f"MSE  = {mse:.2f}")
    print(f"MAE  = {mae:.2f}")
    print(f"R²   = {r2:.4f}")
    print(f"MAPE = {mape*100:.2f}%")



=== GridSearch for Random Forest ===
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best Params: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 300}

=== Random Forest (best) ===
MSE : 5348.135090716165
MAE : 25.332964362486866
R²  : 0.8925970651105655
MAPE: 1.609601564588457e+17 %

=== Random Forest (GridSearch) ===
MSE  = 5348.14
MAE  = 25.33
R²   = 0.8926
MAPE = 160960156458845696.00%

=== Gradient Boosting ===
MSE  = 8404.57
MAE  = 26.62
R²   = 0.8312
MAPE = 165724247312588384.00%

=== XGBoost ===
MSE  = 6607.80
MAE  = 27.25
R²   = 0.8673
MAPE = 161620562816137056.00%


In [5]:
# Không dùng feature mới
# 1. Import thư viện
# ============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

# ============================

# 3. Chọn biến đầu vào (X) và target (y)
# ============================
features = [
    "price", "num_likes", "num_dislikes",
    "restaurant_rating", "num_ratings", "avg_price",
    "restaurant_district", "restaurant_type"
]
X = df[features]
y = df["num_purchases"]

# ============================
# 4. Biến đổi target (log-transform để giảm skew)
# ============================
y_log = np.log1p(y)  # log(1 + y)

# ============================
# 5. Chia dữ liệu train/test
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# ============================
# 6. Tiền xử lý dữ liệu
# ============================
numeric_features = ["price", "num_likes", "num_dislikes",
                    "restaurant_rating", "num_ratings", "avg_price"]

categorical_features = ["restaurant_district", "restaurant_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
# 7. Xây dựng mô hình Random Forest
# ============================
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

# Huấn luyện
model.fit(X_train, y_train)

# ============================
# 8. Dự đoán & đánh giá
# ============================
# Dự đoán trên tập test (log scale)
y_pred_log = model.predict(X_test)

# Chuyển ngược log về giá trị gốc
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

# Tính metric trên giá trị gốc
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred)

print(f"MSE  = {mse:.2f}")
print(f"MAE  = {mae:.2f}")
print(f"R²   = {r2:.4f}")
print(f"MAPE = {mape*100:.2f}%")

# ============================



MSE  = 6211.53
MAE  = 26.19
R²   = 0.8753
MAPE = 160971847845744192.00%


In [6]:
# Không dùng feature mới
# Gradient Boosting Regressor
# ============================
from sklearn.ensemble import GradientBoostingRegressor

# Xây dựng pipeline
model_gb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=200, random_state=42))
])

# Huấn luyện
model_gb.fit(X_train, y_train)

# Dự đoán
y_pred_log_gb = model_gb.predict(X_test)
y_pred_gb = np.expm1(y_pred_log_gb)
y_true_gb = np.expm1(y_test)

# Đánh giá
mse_gb = mean_squared_error(y_true_gb, y_pred_gb)
mae_gb = mean_absolute_error(y_true_gb, y_pred_gb)
r2_gb = r2_score(y_true_gb, y_pred_gb)
mape_gb = mean_absolute_percentage_error(y_true_gb, y_pred_gb)

print("=== Gradient Boosting ===")
print(f"MSE  = {mse_gb:.2f}")
print(f"MAE  = {mae_gb:.2f}")
print(f"R²   = {r2_gb:.4f}")
print(f"MAPE = {mape_gb*100:.2f}%\n")


=== Gradient Boosting ===
MSE  = 13135.08
MAE  = 28.75
R²   = 0.7362
MAPE = 178424315613054240.00%



In [ ]:
# Không dùng feature mới
# XGBoost Regressor
# ============================
from xgboost import XGBRegressor

# Xây dựng pipeline
model_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

# Huấn luyện
model_xgb.fit(X_train, y_train)

# Dự đoán
y_pred_log_xgb = model_xgb.predict(X_test)
y_pred_xgb = np.expm1(y_pred_log_xgb)
y_true_xgb = np.expm1(y_test)

# Đánh giá
mse_xgb = mean_squared_error(y_true_xgb, y_pred_xgb)
mae_xgb = mean_absolute_error(y_true_xgb, y_pred_xgb)
r2_xgb = r2_score(y_true_xgb, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_true_xgb, y_pred_xgb)

print("=== XGBoost ===")
print(f"MSE  = {mse_xgb:.2f}")
print(f"MAE  = {mae_xgb:.2f}")
print(f"R²   = {r2_xgb:.4f}")
print(f"MAPE = {mape_xgb*100:.2f}%\n")


=== XGBoost ===
MSE  = 6181.90
MAE  = 27.09
R²   = 0.8759
MAPE = 163846351926571424.00%

